# RAG Pipeline — Component 1 & 2 (LangChain)

Tests:
- `backend/rag/document_loader.py` — uses LangChain loaders
- `backend/rag/chunker.py` — uses LangChain `RecursiveCharacterTextSplitter`

**LangChain Document object:**
```python
doc.page_content   # the text
doc.metadata       # dict: source, page, type, chunk_index, ...
```

## Setup — add project root to sys.path

In [3]:
import sys, os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root:", project_root)

Project root: c:\Users\VilasTade\OneDrive - IBM\Projects\Practice Projects\End to End Gen AI Project


---
# Component 1 — Document Loader
Module: `backend/rag/document_loader.py`

**LangChain loaders used:**
| File type | LangChain Loader |
|---|---|
| `.pdf` | `PyPDFLoader` — one Document per page |
| `.docx` | `Docx2txtLoader` — one Document for whole file |
| `.txt` | `TextLoader` — one Document for whole file |

In [1]:
from backend.rag.document_loader import load_document

print("✅ document_loader imported")

✅ document_loader imported


## Test 1 — Load TXT file

In [ ]:
txt_path = os.path.join(project_root, "data", "sample_test.txt")
docs = load_document(txt_path)

print(f"Documents returned : {len(docs)}")
print(f"Type of each item  : {type(docs[0])}")
print()
for i, doc in enumerate(docs):
    print(f"--- Doc {i+1} ---")
    print(f"  metadata     : {doc.metadata}")
    print(f"  page_content : {doc.page_content[:300]}" + ("..." if len(doc.page_content) > 300 else ""))
    print()

## Test 2 — Load PDF (drop any PDF into data/ folder first)

In [4]:
pdf_path = os.path.join(project_root, "data/rag_data", "punch-bsvi-09-09-21.pdf")  # <- change filename if needed

if not os.path.exists(pdf_path):
    print(f"⚠️  Not found: {pdf_path}")
    print("   Drop any PDF into data/ and update the filename above.")
else:
    docs = load_document(pdf_path)
    print(f"Pages (Documents) returned: {len(docs)}")
    for doc in docs[:3]:
        print(f"\n  metadata     : {doc.metadata}")
        print(f"  page_content : {doc.page_content[:300]}" + ("..." if len(doc.page_content) > 300 else ""))

Pages (Documents) returned: 532

  metadata     : {'source': 'punch-bsvi-09-09-21.pdf', 'page': 1, 'type': 'pdf', 'content_type': 'text'}
  page_content : Revision: Rev 00/SEPT 2021
OWNER’S MANUAL

  metadata     : {'source': 'punch-bsvi-09-09-21.pdf', 'page': 2, 'type': 'pdf', 'content_type': 'text'}
  page_content : ii
CUSTOMER ASSISTANCE
In our constant endeavour to provide assistance and complete  
service backup, TATA   MOTORS has established an all India cus-
tomer assistance centre.   
In case you have a query regarding any aspect of your vehicle,  
our Customer Assistance Centre will be glad to assist you...

  metadata     : {'source': 'punch-bsvi-09-09-21.pdf', 'page': 3, 'type': 'pdf', 'content_type': 'text'}
  page_content : FOREWORD
iii
Dear Customer,  
  
Welcome to the TATA   MOTORS family.  
  
We congratulate you on the purchase of your new vehicle and are privileged to have you as our valued customer. 
  
We urge you to read this Owner's Manual carefully and familiari

In [6]:
table_docs = [
    doc for doc in docs
    if doc.metadata.get("content_type") == "text"
]

table_docs

[Document(metadata={'source': 'punch-bsvi-09-09-21.pdf', 'page': 1, 'type': 'pdf', 'content_type': 'text'}, page_content='Revision: Rev 00/SEPT 2021\nOWNER’S MANUAL'),
 Document(metadata={'source': 'punch-bsvi-09-09-21.pdf', 'page': 2, 'type': 'pdf', 'content_type': 'text'}, page_content='ii\nCUSTOMER ASSISTANCE\nIn our constant endeavour to provide assistance and complete  \nservice backup, TATA   MOTORS has established an all India cus-\ntomer assistance centre.   \nIn case you have a query regarding any aspect of your vehicle,  \nour Customer Assistance Centre will be glad to assist you on our \nToll Free no. 1800 209 8282\nYou can also approach nearest TATA   MOTORS dealer.  \nA separate Dealer network address booklet is provided with the  \nOwner’s manual. \nTATA MOTORS 24X7 Roadside Assistance Program offers tech-\nnical help in the event of a breakdown. Call the toll-free Road-\nside Assistance.  \nFor additional information, refer to "24X7 Roadside Assis-\ntance" section in the

In [78]:
import os
import io
from pathlib import Path
from typing import List

from langchain_core.documents import Document


def _make_doc(text: str, source: str, page: int, file_type: str,
              content_type: str = "text") -> Document:
    return Document(
        page_content=text.strip(),
        metadata={
            "source": source,
            "page": page,
            "type": file_type,
            "content_type": content_type,   # "text" | "table" | "image"
        }
    )



def _load_pdf(file_path: str) -> List[Document]:
    import pdfplumber
    from langchain_community.document_loaders import PyPDFLoader

    source = os.path.basename(file_path)
    docs: List[Document] = []


    # --- Tables (via pdfplumber) ---
    with pdfplumber.open(file_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            for table in page.extract_tables():
                if not table:
                    continue
                # Convert to markdown table string
                rows = []
                for i, row in enumerate(table):
                    clean = [str(cell or "").replace("\n", " ").strip() for cell in row]
                    rows.append("| " + " | ".join(clean) + " |")
                    if i == 0:
                        rows.append("|" + "|".join(["---"] * len(row)) + "|")
                md_table = "\n".join(rows)
                if md_table.strip():
                    docs.append(_make_doc(md_table, source, page_num, "pdf", "table"))
                    
    return docs


In [79]:
d = _load_pdf(pdf_path)

In [80]:
d

[Document(metadata={'source': 'punch-bsvi-09-09-21.pdf', 'page': 5, 'type': 'pdf', 'content_type': 'table'}, page_content='| 01. SAFETY |  |  |\n|---|---|---|\n|  | Important Information | 1 |\n|  | Seat Belts | 4 |\n|  | Child Restraint System (CRS) | 7 |\n|  | Supplementary Restraint System (srs) (airbags) (if equipped) | 12 |\n|  | Child Lock (if equipped) | 18 |\n|  | Anti-theft Device Immobilizer / PEPS | 19 |\n|  | Anti-lock Braking System (ABS) | 20 |\n|  | Electronic Brake Force Distribution (EBD) | 21 |\n|  | Brake Sway Control (bsc) | 21 |\n|  | Stuck Mode Recovery (traction Pro) (if applica- ble) | 21 |'),
 Document(metadata={'source': 'punch-bsvi-09-09-21.pdf', 'page': 5, 'type': 'pdf', 'content_type': 'table'}, page_content='|  | Doors | 31 |\n|---|---|---|\n|  | Windows | 32 |\n|  | Tail Gate Opening | 36 |\n|  | Fuel Lid | 37 |'),
 Document(metadata={'source': 'punch-bsvi-09-09-21.pdf', 'page': 5, 'type': 'pdf', 'content_type': 'table'}, page_content='| 03. DASHBOARD AND

In [77]:
import pdfplumber
with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):

        tables = page.extract_tables()

        print(f"Page {page_num}: Found {len(tables)} tables")

        for table in tables:
            print(table)


Page 1: Found 0 tables
Page 2: Found 0 tables
Page 3: Found 0 tables
Page 4: Found 0 tables
Page 5: Found 4 tables
[['01. SAFETY', None, None], ['', 'Important Information', '1'], ['', 'Seat Belts', '4'], ['', 'Child Restraint System (CRS)', '7'], ['', 'Supplementary Restraint System (srs) (airbags)\n(if equipped)', '12'], ['', 'Child Lock (if equipped)', '18'], ['', 'Anti-theft Device Immobilizer / PEPS', '19'], ['', 'Anti-lock Braking System (ABS)', '20'], ['', 'Electronic Brake Force Distribution (EBD)', '21'], ['', 'Brake Sway Control (bsc)', '21'], ['', 'Stuck Mode Recovery (traction Pro) (if applica-\nble)', '21']]
[['', 'Doors', '31'], ['', 'Windows', '32'], ['', 'Tail Gate Opening', '36'], ['', 'Fuel Lid', '37']]
[['03. DASHBOARD AND FEATURES', None, None], ['', 'Cockpit', '39'], ['', 'Instrument Cluster (TFT Screen)', '40'], ['', 'Instrument Cluster (LCD Screen)', '56'], ['', 'Tell Tales', '68'], ['', 'Audio Reminders (as available)', '75'], ['', 'Combi-switch (RH Stalk) (if a

## Test 3 — Load DOCX (drop any .docx into data/ folder first)

In [ ]:
docx_path = os.path.join(project_root, "data", "sample.docx")  # <- change filename if needed

if not os.path.exists(docx_path):
    print(f"⚠️  Not found: {docx_path}")
    print("   Drop any .docx into data/ and update the filename above.")
else:
    docs = load_document(docx_path)
    print(f"Documents returned: {len(docs)}")
    for doc in docs:
        print(f"\n  metadata     : {doc.metadata}")
        print(f"  page_content : {doc.page_content[:300]}" + ("..." if len(doc.page_content) > 300 else ""))

## Test 4 — Unsupported file type (should raise ValueError)

In [ ]:
try:
    load_document("report.csv")
except ValueError as e:
    print(f"✅ Correctly raised ValueError: {e}")

---
## Enhanced Component 1 — Tables & Images

The loader now extracts **3 content types** from each document:

| `content_type` | What it contains | How extracted |
|---|---|---|
| `text` | Normal paragraphs | PyPDFLoader / Docx2txtLoader |
| `table` | Tables as markdown strings | pdfplumber (PDF) / python-docx (DOCX) |
| `image` | OCR text from embedded images | pytesseract (requires Tesseract binary) |

### Test A — Load DOCX with a table

In [ ]:
import importlib
import backend.rag.document_loader as dl_module
importlib.reload(dl_module)
from backend.rag.document_loader import load_document

docx_path = os.path.join(project_root, 'data', 'sample_with_table.docx')
docs = load_document(docx_path)

print(f'Total docs extracted : {len(docs)}')
print()

# Group by content_type
from collections import Counter
counts = Counter(d.metadata['content_type'] for d in docs)
print(f'Content type breakdown: {dict(counts)}')
print()

### Test B — Inspect text blocks

In [ ]:
text_docs = [d for d in docs if d.metadata['content_type'] == 'text']
print(f'Text blocks: {len(text_docs)}')
print()
for i, d in enumerate(text_docs):
    print(f'--- Text block {i+1} ---')
    print(f'  metadata     : {d.metadata}')
    print(f'  page_content : {d.page_content[:200]}')
    print()

### Test C — Inspect extracted table (markdown format)

In [ ]:
table_docs = [d for d in docs if d.metadata['content_type'] == 'table']
print(f'Tables found: {len(table_docs)}')
print()
for i, d in enumerate(table_docs):
    print(f'--- Table {i+1} ---')
    print(f'  metadata : {d.metadata}')
    print(f'  Markdown table:')
    print(d.page_content)
    print()

### Test D — Load a PDF with tables (drop one into data/ folder)

Any PDF with a table will work — annual reports, invoices, data sheets, etc.

In [ ]:
pdf_path = os.path.join(project_root, 'data', 'sample.pdf')  # <- change filename if needed

if not os.path.exists(pdf_path):
    print(f'⚠️  Not found: {pdf_path}')
    print('   Drop any PDF into data/ and update the filename above.')
else:
    pdf_docs = load_document(pdf_path)
    counts = Counter(d.metadata['content_type'] for d in pdf_docs)
    print(f'Total extracted: {len(pdf_docs)} | Breakdown: {dict(counts)}')
    print()
    for content_type in ['text', 'table', 'image']:
        typed = [d for d in pdf_docs if d.metadata['content_type'] == content_type]
        if typed:
            print(f'--- First {content_type} block ---')
            print(f'  metadata : {typed[0].metadata}')
            print(f'  content  : {typed[0].page_content[:300]}')
            print()

### Test E — Store all content types in ChromaDB and search

Tables and images are stored just like text — ChromaDB embeds the markdown string or OCR text.

In [ ]:
import importlib
import backend.rag.chunker as ck_module
import backend.rag.vector_store as vs_module
importlib.reload(ck_module)
importlib.reload(vs_module)
from backend.rag.chunker import chunk_documents
from backend.rag.vector_store import clear, add_documents, mmr_search, bm25_search

# Clear and reload
clear(collection_name='test_collection')

# Load the DOCX with table
docs = load_document(os.path.join(project_root, 'data', 'sample_with_table.docx'))
chunks = chunk_documents(docs, chunk_size=500, chunk_overlap=50)

print(f'Docs : {len(docs)} | Chunks : {len(chunks)}')
print(f'Chunk content_types: {dict(Counter(c.metadata["content_type"] for c in chunks))}')
print()

store = add_documents(chunks, collection_name='test_collection')
print('✅ All chunks (text + table) stored in ChromaDB')

### Test F — Query the table data

In [ ]:
query = 'IBM revenue by segment'
results = mmr_search(query, k=3, fetch_k=10, collection_name='test_collection')

print(f"Query: '{query}'")
print()
for i, doc in enumerate(results):
    ctype = doc.metadata.get('content_type', 'unknown')
    print(f'--- Result {i+1} [{ctype}] ---')
    print(doc.page_content)
    print()

### Component 1 — Full Output Summary

| Field | How to access | Description |
|---|---|---|
| Text | `doc.page_content` | Extracted text / markdown table / OCR text |
| Filename | `doc.metadata['source']` | Original filename only |
| Page | `doc.metadata['page']` | 1-based page number |
| File type | `doc.metadata['type']` | `pdf` / `docx` / `txt` |
| Content type | `doc.metadata['content_type']` | `text` / `table` / `image` |

**Table format stored in ChromaDB:**
```
| Segment        | Revenue (USD) | YoY Growth |
|----------------|---------------|------------|
| Software       | 24.4B         | +5.5%      |
| Consulting     | 19.8B         | +4.1%      |
```
The LLM receives this markdown string as context and can answer questions like
'Which segment had the highest revenue?' directly from the table.

### Component 1 — Output Summary

| Field | How to access | Description |
|---|---|---|
| Text | `doc.page_content` | Extracted text |
| Filename | `doc.metadata['source']` | Original filename only |
| Page | `doc.metadata['page']` | 1-based page number |
| Type | `doc.metadata['type']` | `pdf` / `docx` / `txt` |

---
# Component 2 — Chunker
Module: `backend/rag/chunker.py`

**LangChain class used:** `RecursiveCharacterTextSplitter`

Tries to split at natural boundaries in this order:
```
paragraph (\n\n)  →  line (\n)  →  sentence (. )  →  word ( )  →  character
```
So chunks never break mid-word unless absolutely necessary.

In [ ]:
from backend.rag.chunker import chunk_documents

print("✅ chunker imported")

## Test 1 — Chunk the TXT file (small size to see overlap clearly)

In [ ]:
txt_path = os.path.join(project_root, "data", "sample_test.txt")
docs = load_document(txt_path)

# Use small chunk_size so we get multiple chunks from the sample file
chunks = chunk_documents(docs, chunk_size=200, chunk_overlap=50)

print(f"Input documents : {len(docs)}")
print(f"Output chunks   : {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(f"  metadata     : {chunk.metadata}")
    print(f"  page_content : {chunk.page_content}")
    print()

## Test 2 — See the overlap (tail of chunk N vs head of chunk N+1)

In [ ]:
if len(chunks) >= 2:
    for i in range(len(chunks) - 1):
        tail = chunks[i].page_content[-60:]
        head = chunks[i+1].page_content[:60]
        print(f"Chunk {i+1} tail : ...'{tail}'")
        print(f"Chunk {i+2} head :    '{head}'...")
        print()
else:
    print("Only 1 chunk — try a smaller chunk_size.")

## Test 3 — RecursiveCharacterTextSplitter splits at natural boundaries

Compare with a naive character splitter — LangChain's splitter won't cut mid-sentence.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

sample_text = "IBM was founded in 1911. It operates in 175 countries. Revenue was $61.9 billion in 2023. The watsonx platform powers enterprise AI."

# LangChain RecursiveCharacterTextSplitter
recursive = RecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=10)
r_chunks = recursive.split_text(sample_text)

print("=== RecursiveCharacterTextSplitter (chunk_size=60, overlap=10) ===")
for i, c in enumerate(r_chunks):
    print(f"  Chunk {i+1}: '{c}'")

print()

# Naive CharacterTextSplitter (splits only on separator)
naive = CharacterTextSplitter(chunk_size=60, chunk_overlap=10, separator=" ")
n_chunks = naive.split_text(sample_text)

print("=== CharacterTextSplitter — naive (chunk_size=60, overlap=10) ===")
for i, c in enumerate(n_chunks):
    print(f"  Chunk {i+1}: '{c}'")

## Test 4 — Production settings (chunk_size=1000, overlap=200)

In [ ]:
chunks_prod = chunk_documents(docs, chunk_size=1000, chunk_overlap=200)

print(f"chunk_size=1000, chunk_overlap=200")
print(f"  Input docs : {len(docs)}")
print(f"  Chunks     : {len(chunks_prod)}")
print()
for i, c in enumerate(chunks_prod):
    print(f"  Chunk {i+1}: {len(c.page_content)} chars | metadata: {c.metadata}")

### Component 2 — Output Summary

| Field | How to access | Description |
|---|---|---|
| Chunk text | `chunk.page_content` | The chunk text (≤ chunk_size) |
| Filename | `chunk.metadata['source']` | Original filename |
| Page | `chunk.metadata['page']` | Page it came from |
| Chunk # | `chunk.metadata['chunk_index']` | Which chunk on that page |
| Start pos | `chunk.metadata['start_index']` | Char offset in the original text |

This output feeds directly into **Component 3 — Embedder + ChromaDB**.

---
# Component 3 — Embedder + ChromaDB Vector Store
Module: `backend/rag/vector_store.py`

**LangChain classes used:**
| Class | Purpose |
|---|---|
| `HuggingFaceEmbeddings` | Converts text → vector using `BAAI/bge-small-en-v1.5` |
| `Chroma` (langchain-chroma) | Local persistent vector database |

**Key operations:**
- `add_documents(chunks)` — embed and store chunks in ChromaDB
- `search(query, k=5)` — find top-K most similar chunks
- `search_with_scores(query)` — same + similarity score
- `clear()` — wipe the collection

**Storage location:** `chroma_db/` folder in project root (persists on disk)

In [ ]:
from backend.rag.vector_store import add_documents, search, search_with_scores, clear, get_embeddings

print("✅ vector_store imported")

## Test 1 — Load the embedding model

First time: downloads `BAAI/bge-small-en-v1.5` (~130 MB). Cached locally after that.

In [ ]:
embeddings = get_embeddings()

# Embed a single sentence to verify the model works
test_vector = embeddings.embed_query("What is IBM revenue?")

print(f"✅ Embedding model loaded: BAAI/bge-small-en-v1.5")
print(f"   Vector dimensions : {len(test_vector)}")
print(f"   First 5 values    : {[round(v, 4) for v in test_vector[:5]]}")

## Test 2 — Embed two sentences and compare similarity

Similar sentences should have a high cosine similarity (close to 1.0).

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

s1 = "IBM revenue grew in 2023"
s2 = "IBM reported higher earnings last year"  # similar meaning
s3 = "The weather in Paris is sunny today"     # unrelated

v1 = embeddings.embed_query(s1)
v2 = embeddings.embed_query(s2)
v3 = embeddings.embed_query(s3)

print(f"Sentence 1 : '{s1}'")
print(f"Sentence 2 : '{s2}'")
print(f"Sentence 3 : '{s3}'")
print()
print(f"Similarity(s1, s2) = {cosine_similarity(v1, v2):.4f}  ← should be HIGH (similar meaning)")
print(f"Similarity(s1, s3) = {cosine_similarity(v1, v3):.4f}  ← should be LOW  (unrelated)")

## Test 3 — Store chunks in ChromaDB

Load → Chunk → Store. This is the full ingestion pipeline.

In [ ]:
# Clear any previous test data first
clear(collection_name="test_collection")

# Load + Chunk the sample file
txt_path = os.path.join(project_root, "data", "sample_test.txt")
docs   = load_document(txt_path)
chunks = chunk_documents(docs, chunk_size=200, chunk_overlap=50)

print(f"Chunks to store: {len(chunks)}")

# Embed and store in ChromaDB
store = add_documents(chunks, collection_name="test_collection")

print(f"✅ Stored {len(chunks)} chunks in ChromaDB collection 'test_collection'")
print(f"   Persisted at: chroma_db/ in project root")

## Test 4 — Search ChromaDB (similarity search)

Ask a question — ChromaDB finds the most relevant chunks.

In [ ]:
query = "What is IBM revenue?"
results = search(query, k=3, collection_name="test_collection")

print(f"Query  : '{query}'")
print(f"Top {len(results)} results:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"  metadata     : {doc.metadata}")
    print(f"  page_content : {doc.page_content}")
    print()

## Test 5 — Search with similarity scores

In [ ]:
query = "watsonx AI platform"
results_with_scores = search_with_scores(query, k=3, collection_name="test_collection")

print(f"Query: '{query}'")
print(f"Results with scores (lower L2 distance = more similar):\n")
for i, (doc, score) in enumerate(results_with_scores):
    print(f"--- Result {i+1} | score: {score:.4f} ---")
    print(f"  page_content : {doc.page_content}")
    print()

## Test 6 — Confirm ChromaDB persists on disk

In [ ]:
import os

chroma_path = os.path.join(project_root, "chroma_db")
print(f"ChromaDB path: {chroma_path}")
print(f"Exists: {os.path.exists(chroma_path)}")

if os.path.exists(chroma_path):
    for f in os.listdir(chroma_path):
        print(f"  {f}")

## Test 4 — MMR Search (relevant + diverse)

`lambda_mult=0.5` → balanced relevance + diversity  
`lambda_mult=1.0` → pure relevance (same as similarity search)  
`lambda_mult=0.0` → maximum diversity

In [ ]:
from backend.rag.vector_store import mmr_search, bm25_search, hybrid_search, search_with_scores

query = "What is IBM revenue?"

mmr_results = mmr_search(query, k=3, fetch_k=10, lambda_mult=0.5, collection_name="test_collection")

print(f"Query : '{query}'")
print(f"MMR results (k=3, fetch_k=10, lambda_mult=0.5):\n")
for i, doc in enumerate(mmr_results):
    print(f"--- Result {i+1} ---")
    print(f"  metadata     : {doc.metadata}")
    print(f"  page_content : {doc.page_content}")
    print()

## Test 5 — MMR vs Plain Similarity (side-by-side)

See how MMR avoids returning near-duplicate chunks that plain similarity would return.

In [ ]:
query = "IBM revenue and earnings"

# Plain similarity (may return redundant chunks)
sim_results = search_with_scores(query, k=3, collection_name="test_collection")

# MMR (diverse results)
mmr_results = mmr_search(query, k=3, fetch_k=10, lambda_mult=0.5, collection_name="test_collection")

print("=== Plain Similarity Search ===")
for i, (doc, score) in enumerate(sim_results):
    print(f"  [{i+1}] score={score:.4f} | {doc.page_content[:80]}")

print()
print("=== MMR Search (lambda=0.5) ===")
for i, doc in enumerate(mmr_results):
    print(f"  [{i+1}] {doc.page_content[:80]}")

## Test 6 — lambda_mult effect on MMR

See how changing `lambda_mult` shifts results from diverse → relevant.

In [ ]:
query = "IBM AI strategy"

for lam in [0.0, 0.5, 1.0]:
    results = mmr_search(query, k=3, fetch_k=10, lambda_mult=lam, collection_name="test_collection")
    print(f"--- lambda_mult={lam} ({'max diversity' if lam==0 else 'max relevance' if lam==1 else 'balanced'}) ---")
    for i, doc in enumerate(results):
        print(f"  [{i+1}] {doc.page_content[:80]}")
    print()

## Test 7 — BM25 Keyword Search

In [ ]:
query = "watsonx"

bm25_results = bm25_search(query, chunks, k=3)

print(f"Query : '{query}'")
print(f"BM25 results (keyword match):\n")
for i, doc in enumerate(bm25_results):
    print(f"--- Result {i+1} ---")
    print(f"  metadata     : {doc.metadata}")
    print(f"  page_content : {doc.page_content}")
    print()

## Test 8 — Hybrid Search (BM25 + MMR via Reciprocal Rank Fusion)

In [ ]:
query = "IBM revenue and watsonx AI"

hybrid_results = hybrid_search(
    query,
    chunks,
    k=3,
    collection_name="test_collection",
    mmr_weight=0.6,
    bm25_weight=0.4,
    lambda_mult=0.5
)

print(f"Query : '{query}'")
print(f"Hybrid results (MMR 60% + BM25 40%):\n")
for i, doc in enumerate(hybrid_results):
    print(f"--- Result {i+1} ---")
    print(f"  metadata     : {doc.metadata}")
    print(f"  page_content : {doc.page_content}")
    print()

## Test 9 — Compare all 3 retrievers side-by-side

In [ ]:
query = "IBM watsonx revenue consulting"

bm25_r   = bm25_search(query, chunks, k=3)
mmr_r    = mmr_search(query, k=3, fetch_k=10, lambda_mult=0.5, collection_name="test_collection")
hybrid_r = hybrid_search(query, chunks, k=3, collection_name="test_collection")

print(f"Query: '{query}'\n")

print("=== BM25 (keyword) ===")
for i, d in enumerate(bm25_r):   print(f"  [{i+1}] {d.page_content[:90]}")

print("\n=== MMR (semantic + diverse) ===")
for i, d in enumerate(mmr_r):    print(f"  [{i+1}] {d.page_content[:90]}")

print("\n=== Hybrid (BM25 + MMR) ===")
for i, d in enumerate(hybrid_r): print(f"  [{i+1}] {d.page_content[:90]}")

### Component 3 — Full Output Summary

| Retriever | Function | Best for |
|---|---|---|
| **MMR** | `mmr_search(query, k, fetch_k, lambda_mult)` | Diverse, non-redundant semantic results |
| **BM25** | `bm25_search(query, chunks, k)` | Exact keywords, product codes, numbers |
| **Hybrid** | `hybrid_search(query, chunks, k, ...)` | Best of both — used in production RAG |
| **Similarity+score** | `search_with_scores(query, k)` | Debugging / score inspection |

**`lambda_mult` guide:**
| Value | Behaviour |
|---|---|
| `0.0` | Maximum diversity — picks most different chunks |
| `0.5` | Balanced (recommended default) |
| `1.0` | Pure relevance — same as plain similarity search |

**Full pipeline so far:**
```
File
 └─ load_document()      → List[Document]              (Component 1)
     └─ chunk_documents() → List[Document] chunks       (Component 2)
         └─ add_documents() → stored in ChromaDB        (Component 3)
         └─ mmr_search()    → relevant + diverse chunks
         └─ bm25_search()   → keyword matched chunks
         └─ hybrid_search() → best of both              ← use this in RAG
```

This output feeds directly into **Component 4 — Reranker**.

---
# Component 4 — Reranker
Module: `backend/rag/reranker.py`

**Model:** `BAAI/bge-reranker-base` (Cross-Encoder via `sentence-transformers`)

### Why reranking?
The retriever (MMR/BM25/Hybrid) fetches candidates **fast** using approximate vector search.
The reranker reads the **query + chunk together** for a much more accurate relevance score.

```
Bi-encoder  (retriever) : query → vector,  chunk → vector  → cosine distance  (fast, approximate)
Cross-encoder (reranker): [query + chunk]  → one score      (slow, accurate)
```

**Typical pattern:**
```
hybrid_search(query, k=10)   ← fetch 10 candidates fast
    └── rerank(query, docs, top_k=3)  ← score all 10, return best 3
```

⚠️ First run downloads `BAAI/bge-reranker-base` (~1.1 GB). Cached after that.

In [ ]:
import importlib
import backend.rag.reranker as rr_module
importlib.reload(rr_module)
from backend.rag.reranker import get_reranker, rerank

print('✅ reranker imported')

## Test 1 — Load the reranker model

First run downloads `bge-reranker-base` (~1.1 GB). Watch the progress bar.

In [ ]:
reranker = get_reranker()
print(f'✅ Reranker loaded: {reranker.model.name_or_path}')

# Score a single (query, text) pair manually
score = reranker.predict([('What is IBM revenue?', 'IBM reported $61.9 billion in revenue in 2023.')])
print(f'   Score for relevant pair  : {score[0]:.4f}  ← should be HIGH')

score2 = reranker.predict([('What is IBM revenue?', 'The weather in Paris is sunny today.')])
print(f'   Score for irrelevant pair: {score2[0]:.4f}  ← should be LOW')

## Test 2 — Rerank retrieved chunks

First retrieve candidates with hybrid search, then rerank them.

In [ ]:
import importlib
import backend.rag.vector_store as vs_module
importlib.reload(vs_module)
from backend.rag.vector_store import hybrid_search, clear, add_documents
from backend.rag.chunker import chunk_documents
from backend.rag.document_loader import load_document
from collections import Counter

# Make sure test_collection has data
clear(collection_name='test_collection')
docs   = load_document(os.path.join(project_root, 'data', 'sample_with_table.docx'))
chunks = chunk_documents(docs, chunk_size=500, chunk_overlap=50)
add_documents(chunks, collection_name='test_collection')
print(f'Stored {len(chunks)} chunks')

In [ ]:
query = 'IBM revenue by segment'

# Step 1: Retrieve top-10 candidates
candidates = hybrid_search(query, chunks, k=10, collection_name='test_collection')
print(f'Candidates from hybrid search: {len(candidates)}')
print()
print('Before reranking (retriever order):')
for i, doc in enumerate(candidates):
    ctype = doc.metadata.get('content_type', '')
    print(f'  [{i+1}] [{ctype}] {doc.page_content[:80]}')

In [ ]:
# Step 2: Rerank → keep top 3
reranked = rerank(query, candidates, top_k=3)

print(f'After reranking (top 3):')
print()
for i, doc in enumerate(reranked):
    ctype  = doc.metadata.get('content_type', '')
    score  = doc.metadata.get('rerank_score', 0)
    print(f'--- Result {i+1} | rerank_score={score} | [{ctype}] ---')
    print(f'  {doc.page_content}')
    print()

## Test 3 — See how reranking reorders results

Compare the order BEFORE and AFTER reranking.

In [ ]:
query = 'watsonx AI strategy'

candidates = hybrid_search(query, chunks, k=6, collection_name='test_collection')
reranked   = rerank(query, candidates, top_k=6)

print(f'Query: "{query}"\n')
print(f'{"Rank":<6} {"Before (retriever)":<55} {"After (reranker)"}')
print('-' * 120)
for i in range(len(candidates)):
    before = candidates[i].page_content[:50].replace('\n',' ')
    after  = reranked[i].page_content[:50].replace('\n',' ')
    score  = reranked[i].metadata.get('rerank_score', '')
    moved  = '←moved' if candidates[i].page_content != reranked[i].page_content else ''
    print(f'{i+1:<6} {before:<55} {after}  score={score}  {moved}')

## Test 4 — Full RAG retrieval pipeline end-to-end

This is the complete flow that will be used in production:
```
User query
  └─ hybrid_search(k=10)   → 10 candidates
      └─ rerank(top_k=3)   → 3 best, accurately scored
          └─ [feed to LLM as context]
```

In [ ]:
def retrieve(query: str, chunks, collection_name='test_collection', fetch_k=10, final_k=3):
    """Full retrieval pipeline: Hybrid search → Rerank → return top final_k"""
    candidates = hybrid_search(query, chunks, k=fetch_k, collection_name=collection_name)
    return rerank(query, candidates, top_k=final_k)

# Test with different queries
queries = [
    'What is IBM total revenue?',
    'Which segment had the highest growth?',
    'Tell me about watsonx platform',
]

for q in queries:
    results = retrieve(q, chunks)
    print(f"Query: '{q}'")
    for i, doc in enumerate(results):
        score = doc.metadata.get('rerank_score', 0)
        ctype = doc.metadata.get('content_type', '')
        print(f"  [{i+1}] score={score} [{ctype}] {doc.page_content[:90].replace(chr(10),' ')}")
    print()

### Component 4 — Output Summary

| Function | Input | Output |
|---|---|---|
| `get_reranker()` | — | Loaded `CrossEncoder` model |
| `rerank(query, docs, top_k)` | query + candidate docs | Top-K docs sorted by `rerank_score` |

**Each result doc now has:**
```python
doc.page_content              # chunk text
doc.metadata['rerank_score']  # float — higher = more relevant
doc.metadata['content_type']  # 'text' / 'table' / 'image'
doc.metadata['source']        # original filename
doc.metadata['page']          # page number
```

**Complete pipeline:**
```
File
 └─ load_document()      → text + tables + images    (Component 1)
     └─ chunk_documents() → overlapping chunks        (Component 2)
         └─ add_documents() → stored in ChromaDB      (Component 3)
         └─ hybrid_search(k=10) → 10 candidates
             └─ rerank(top_k=3) → 3 best chunks       (Component 4)
                 └─ [fed to LLM as context]            (next: Component 5)
```